# Spike E00.5 — kernel Java de Jupyter contra JDK 25

De-riesgo para [#15](https://github.com/InaDarta/promtior-challenge/issues/15): probar `rapaio-jupyter-kernel` contra el JDK 25 del proyecto, verificar que la magia de dependencias Maven resuelve un artefacto de LangChain4j, e instanciar un `ChatModel` con una llamada real.

**Prerequisitos, corridos en tu PowerShell normal (no en la tool de Claude Code):**

Necesitás una API key de [Google AI Studio](https://aistudio.google.com/apikey) para
`GEMINI_API_KEY` (o, si preferís Groq, `GROQ_API_KEY` -- sacala en
[Groq Console](https://console.groq.com/keys); ver `E09.4-agente-de-reservas.ipynb` para el
failover entre las dos).

```powershell
$env:JAVA_HOME = "C:\Program Files\Eclipse Adoptium\jdk-25.0.4.101-hotspot"
$env:Path = "$env:JAVA_HOME\bin;$env:Path"
java -version   # confirmar que da 25.x

pip install jupyterlab   # si todavía no lo tenés

# Descargar el jar del kernel (release 3.0.4):
# https://github.com/padreati/rapaio-jupyter-kernel/releases/download/3.0.4/rapaio-jupyter-kernel-3.0.4.jar
java -jar rapaio-jupyter-kernel-3.0.4.jar -i -auto -preview25

$env:GEMINI_API_KEY = "tu-api-key"   # o $env:GROQ_API_KEY -- ver de dónde sacarlas más arriba
jupyter lab
```

Abrí este notebook desde Jupyter Lab, elegí el kernel Java instalado (`rapaio-jupyter-kernel`) y corré las celdas de abajo en orden.

## 1. Magia de dependencias Maven: resolver LangChain4j

In [1]:
%dependency /add dev.langchain4j:langchain4j-google-ai-gemini:1.0.0-beta5
%dependency /resolve

Adding dependency dev.langchain4j:langchain4j-google-ai-gemini:1.0.0-beta5
Solving dependencies
Resolved artifacts count: 7
Add to classpath: C:\Users\idartayete\AppData\Roaming\jupyter\kernels\rapaio-jupyter-kernel-preview25\mima_cache\dev\langchain4j\langchain4j-google-ai-gemini\1.0.0-beta5\langchain4j-google-ai-gemini-1.0.0-beta5.jar
Add to classpath: C:\Users\idartayete\AppData\Roaming\jupyter\kernels\rapaio-jupyter-kernel-preview25\mima_cache\dev\langchain4j\langchain4j-core\1.0.0\langchain4j-core-1.0.0.jar
Add to classpath: C:\Users\idartayete\AppData\Roaming\jupyter\kernels\rapaio-jupyter-kernel-preview25\mima_cache\org\slf4j\slf4j-api\2.0.17\slf4j-api-2.0.17.jar
Add to classpath: C:\Users\idartayete\AppData\Roaming\jupyter\kernels\rapaio-jupyter-kernel-preview25\mima_cache\org\jspecify\jspecify\1.0.0\jspecify-1.0.0.jar
Add to classpath: C:\Users\idartayete\AppData\Roaming\jupyter\kernels\rapaio-jupyter-kernel-preview25\mima_cache\com\fasterxml\jackson\core\jackson-annotations\2

## 2. Instanciar un `ChatModel` y hacer una llamada trivial

In [ ]:
import dev.langchain4j.model.chat.ChatModel;
import dev.langchain4j.model.googleai.GoogleAiGeminiChatModel;

String apiKey = System.getenv("GEMINI_API_KEY");
if (apiKey == null || apiKey.isBlank()) {
    throw new IllegalStateException("Falta GEMINI_API_KEY en el entorno");
}

ChatModel model = GoogleAiGeminiChatModel.builder()
        .apiKey(apiKey)
        .modelName("gemini-3.7-flash")
        .build();

String respuesta = model.chat("Respond with exactly the word: pong");
System.out.println(respuesta);

## Resultado del spike

- Kernel arrancó contra JDK 25: ✅
- `%dependency /resolve` bajó LangChain4j (`langchain4j-google-ai-gemini:1.0.0-beta5` + transitivas) sin error: ✅
- La llamada al `ChatModel` devolvió una respuesta: ✅ (`pong`, con `gemini-3.7-flash` — `gemini-2.5-flash` está retirado y `gemini-3.6-flash`, el reemplazo que sugería el 404, ya fue superado; ver [ADR 0002](../doc/adr/0002-notebook-java-o-python.md))

**Decisión**: notebook en Java. Ver [ADR 0002](../doc/adr/0002-notebook-java-o-python.md).